# Deduplicación de Clientes - TABLA_MASTER_ID

## Objetivo
Identificar clientes duplicados en la base de datos usando un algoritmo de grafos basado en Email y Celular.

## Metodología
1. Cargar clientes desde SQL Server
2. Limpiar y normalizar datos (email, celular)
3. Validar calidad de datos
4. Crear grafo de conectividad (si comparten email o celular, están conectados)
5. Identificar componentes conectadas = clientes reales únicos
6. Asignar ID único a cada cliente real
7. Generar reporte con métricas de duplicados

## Output
- Archivo Excel con tabla de clientes deduplicados
- Reporte de estadísticas y duplicados encontrados

## 1. Importar Librerías

In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine, text
import urllib
import networkx as nx
import os
from dotenv import load_dotenv
from datetime import datetime

# Cargar variables de entorno
load_dotenv()

# # Configurar opciones de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Conexión a SQL Server

In [2]:
# CONEXIÓN A SQL SERVER

try:
    params = urllib.parse.quote_plus(
        "DRIVER=ODBC Driver 17 for SQL Server;"
        f"SERVER={os.getenv('DB_SERVER')};"
        "DATABASE=Ventas_Comerssia;"
        f"UID={os.getenv('DB_USER')};"
        f"PWD={os.getenv('DB_PASSWORD')};"
    )

    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    
    # Validar conexión
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@version"))
        version = result.fetchone()[0]
    
    print("✓ Conexión a SQL Server exitosa")
    print(f"  Servidor: {os.getenv('DB_SERVER')}")
    print(f"  Base de datos: Ventas_Comerssia")

except Exception as e:
    print(f"✗ Error al conectar a SQL Server: {str(e)}")
    print("  Verifica las variables de entorno en .env")
    raise

✓ Conexión a SQL Server exitosa
  Servidor: 181.57.189.150,34276
  Base de datos: Ventas_Comerssia


## 3. Cargar y Explorar Datos

In [3]:
# CARGAR DATOS

query = """
SELECT 
    Cliente,
    Documento,
    Nombres,
    Apellidos,
    Email,
    Celular
FROM dbo.Clientes
"""

df = pd.read_sql(query, engine)

print(f"✓ Datos cargados")
print(f"  Registros totales: {len(df):,}")
print(f"\n📊 Primeras filas:")
print(df.head(10))

✓ Datos cargados
  Registros totales: 420,017

📊 Primeras filas:
      Cliente  Documento                   Nombres  \
0   C56056602   56056602                DORIS NELA   
1   C76308467   76308467            JAVIER GERARDO   
2  C901337200  901337200  INFRA LOGISTIC GROUP SAS   
3   C16764205   16764205               JOSE RAMIRO   
4  C900407222  900407222                 FUNDACION   
5   C30882445   30882445                   LISSETE   
6  CB00397475  B00397475      MARIA DE LOS ANGELES   
7   C53080288   53080288             LIZETH MARINA   
8   C79147250   79147250                  MAURICIO   
9   C98523861   98523861              JUAN ALBERTO   

                      Apellidos                           Email     Celular  
0                        SOLANO           orismaria25@gmail.com  1111111111  
1                   RUIZ RIVERA               javiriv@gmail.com  3012561131  
2                           SAS   infra.logisticgroup@gmail.com  3142188153  
3                GOMEZ ENRIQ

## 4. Análisis de Datos Faltantes

In [4]:
# ANÁLISIS DE VALORES FALTANTES

print("\n📋 Valores Faltantes:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Columna': df.columns,
    'Nulos': missing.values,
    '% Nulos': missing_pct.values
})

print(missing_df.to_string(index=False))

print("\n📊 Estadísticas por Columna:")
print(df.describe(include='all').to_string())


📋 Valores Faltantes:


  Columna  Nulos  % Nulos
  Cliente      0     0.00
Documento      0     0.00
  Nombres   1084     0.26
Apellidos  14144     3.37
    Email      0     0.00
  Celular      0     0.00

📊 Estadísticas por Columna:


            Cliente   Documento Nombres Apellidos                 Email     Celular
count        420017      420017  418933    405873                420017      420017
unique       420016      420016   73280    157755                275031      297667
top     C1098660341  1098660341   MARIA     GOMEZ  negado@provenzal.net  1111111111
freq              2           2    7673      2951                134471      104011


## 5. Limpieza y Normalización de Datos

In [5]:
# LIMPIEZA DE DATOS

# Crear copia para trabajar
df_clean = df.copy()

# Sentinel: valor que representa "sin dato válido" — se excluye del grafo
# Todo email o celular técnicamente inválido se normaliza a este valor
SENTINEL_EMAIL = 'negado@provenzal.net'
SENTINEL_PHONE = '1111111111'

# Emails que deben normalizarse al sentinel (genéricos, vacíos, malformados del CRM)
INVALID_EMAILS = [
    'negado@provenzal.net',
    'negado@provenzal.com',
    'noreporta@provenzal.net',
    'notiene@provenzal.net',
    ''
]

# Celulares que deben normalizarse al sentinel
INVALID_PHONES = [
    '1111111111',
    'nan',
    '',
    '0000000000',
    '9999999999'
]

# Umbral: celulares que aparecen en más de N registros se excluyen del grafo
# pero se conservan tal como están en el output (no se reemplazan por sentinel)
MAX_CEL_FREQ = 4

# --- EMAIL ---
df_clean['Email'] = df_clean['Email'].astype(str).str.lower().str.strip()
# Genéricos y vacíos conocidos → sentinel
df_clean.loc[df_clean['Email'].isin(INVALID_EMAILS), 'Email'] = SENTINEL_EMAIL
# Sin formato válido (correo@dominio.ext) → sentinel
email_regex = r'^[\w\.\+\-]+@[\w\-]+\.[a-z]{2,}$'
df_clean.loc[~df_clean['Email'].str.match(email_regex, na=False), 'Email'] = SENTINEL_EMAIL

# --- CELULAR ---
df_clean['Celular'] = df_clean['Celular'].astype(str).str.strip()
# Mantener solo dígitos
df_clean['Celular'] = df_clean['Celular'].apply(lambda x: re.sub(r'[^0-9]', '', str(x)) if x else '')
# Inválidos conocidos → sentinel
df_clean.loc[df_clean['Celular'].isin(INVALID_PHONES), 'Celular'] = SENTINEL_PHONE
# Sin longitud válida (10 dígitos) → sentinel
df_clean.loc[df_clean['Celular'].str.len() != 10, 'Celular'] = SENTINEL_PHONE

# Celulares de alta frecuencia: se conserva el número original pero se excluyen del grafo
cel_freq = df_clean['Celular'].value_counts()
cels_alta_frecuencia = set(
    c for c in cel_freq[cel_freq > MAX_CEL_FREQ].index if c != SENTINEL_PHONE
)

# Conjunto completo de celulares excluidos del grafo
CELS_EXCLUIDOS = {SENTINEL_PHONE} | cels_alta_frecuencia

# --- NOMBRES ---
df_clean['Nombres'] = df_clean['Nombres'].astype(str).str.upper().str.strip()

print("✓ Datos limpios y normalizados")
print("\n📊 Cambios después de limpieza:")
emails_validos = (df_clean['Email'] != SENTINEL_EMAIL).sum()
celulares_validos = (~df_clean['Celular'].isin(CELS_EXCLUIDOS)).sum()
print(f"  Emails válidos:    {emails_validos:,} ({emails_validos/len(df_clean)*100:.1f}%)")
print(f"  Celulares válidos: {celulares_validos:,} ({celulares_validos/len(df_clean)*100:.1f}%)")
print(f"  Celulares excluidos por alta frecuencia (>{MAX_CEL_FREQ} registros): {len(cels_alta_frecuencia):,} números únicos")

print("\n📊 Estadísticas por Columna (datos limpios):")
print(df_clean.describe(include='all').to_string())

✓ Datos limpios y normalizados

📊 Cambios después de limpieza:
  Emails válidos:    274,045 (65.2%)
  Celulares válidos: 313,509 (74.6%)
  Celulares excluidos por alta frecuencia (>4 registros): 201 números únicos

📊 Estadísticas por Columna (datos limpios):


            Cliente   Documento Nombres Apellidos                 Email     Celular
count        420017      420017  420017    405873                420017      420017
unique       420016      420016   70159    157755                264517      297348
top     C1098660341  1098660341   MARIA     GOMEZ  negado@provenzal.net  1111111111
freq              2           2    7886      2951                145972      105275


## 6. Crear Grafo de Conectividad

In [6]:
# CREAR GRAFO DE CONECTIVIDAD

G = nx.Graph()

# Máscaras de datos válidos para el grafo
email_valid_mask = df_clean['Email'] != SENTINEL_EMAIL
cel_valid_mask = ~df_clean['Celular'].isin(CELS_EXCLUIDOS)

# Agregar nodos (todos los clientes)
for cliente in df_clean['Cliente']:
    G.add_node(cliente)

# Agrupar por EMAIL y crear bordes (prioridad principal)
email_groups = df_clean[email_valid_mask].groupby('Email')['Cliente'].apply(list)
email_duplicados = 0

for email, clientes in email_groups.items():
    if len(clientes) > 1:
        email_duplicados += len(clientes) - 1
        for i in range(len(clientes) - 1):
            G.add_edge(clientes[i], clientes[i + 1])

# Frecuencia global de email válido para validar unicidad global
audit_email_freq = df_clean[email_valid_mask].groupby('Email').size()

# Agrupar por CELULAR con regla híbrida
# Solo celulares no excluidos. Une cuando:
# - email sentinel (sin dato válido), o
# - email válido pero único globalmente (aparece en 1 solo registro)
cel_groups = df_clean[cel_valid_mask].groupby('Celular')
cel_duplicados = 0
cel_uniones_email_unico = 0
cel_grupos_email_unico = 0
cel_grupos_con_sentinel = 0

for cel, group in cel_groups:
    if len(group) <= 1:
        continue

    # Unir subgrupos con el mismo email válido dentro del celular
    group_email_valido = group[group['Email'] != SENTINEL_EMAIL]
    for _, sub in group_email_valido.groupby('Email'):
        sub_clientes = sub['Cliente'].tolist()
        if len(sub_clientes) > 1:
            cel_duplicados += len(sub_clientes) - 1
            for i in range(len(sub_clientes) - 1):
                G.add_edge(sub_clientes[i], sub_clientes[i + 1])

    # Unir por celular cuando email es sentinel o único globalmente
    mask_union_cel = (
        group['Email'].eq(SENTINEL_EMAIL)
        | (
            (group['Email'] != SENTINEL_EMAIL)
            & group['Email'].map(audit_email_freq).eq(1)
        )
    )
    clientes_union_cel = group.loc[mask_union_cel, 'Cliente'].tolist()

    if len(clientes_union_cel) > 1:
        cel_grupos_email_unico += 1
        cel_uniones_email_unico += len(clientes_union_cel) - 1
        cel_duplicados += len(clientes_union_cel) - 1

        if group['Email'].eq(SENTINEL_EMAIL).any():
            cel_grupos_con_sentinel += 1

        for i in range(len(clientes_union_cel) - 1):
            G.add_edge(clientes_union_cel[i], clientes_union_cel[i + 1])

print("✓ Grafo creado")
print(f"  Nodos: {G.number_of_nodes():,}")
print(f"  Bordes por email:   {email_duplicados:,}")
print(f"  Bordes por celular: {cel_duplicados:,}")
print(f"  Uniones por celular (email sentinel o único global): {cel_uniones_email_unico:,}")
print(f"  Grupos de celular que activaron esa regla: {cel_grupos_email_unico:,}")
print(f"  Grupos de celular con al menos un email sentinel: {cel_grupos_con_sentinel:,}")

✓ Grafo creado
  Nodos: 420,016
  Bordes por email:   9,529
  Bordes por celular: 15,347
  Uniones por celular (email sentinel o único global): 9,762
  Grupos de celular que activaron esa regla: 8,970
  Grupos de celular con al menos un email sentinel: 4,638


## 7. Identificar Clientes Únicos Reales

In [7]:
# CLUSTERS = CLIENTES REALES ÚNICOS (LÓGICA INCREMENTAL)

componentes = list(nx.connected_components(G))

# Intentar cargar mapeo existente desde SQL
try:
    df_existing = pd.read_sql("SELECT Cliente, ID_Cliente_Unico FROM MASTER_ID", engine)
    map_existente = dict(zip(df_existing['Cliente'], df_existing['ID_Cliente_Unico']))
    max_id_existente = int(df_existing['ID_Cliente_Unico'].max())
    print(f"✓ Mapeo existente cargado: {len(map_existente):,} clientes, max ID: {max_id_existente:,}")
except Exception:
    map_existente = {}
    max_id_existente = 99999
    print("ℹ️  No se encontró tabla MASTER_ID — primera ejecución, asignando IDs desde 100000")

# Asignar IDs estables
map_cliente = {}
grupos_nuevos = 0
fusiones = []
next_id = max_id_existente + 1

for comp in componentes:
    ids_existentes = {map_existente[c] for c in comp if c in map_existente}

    if ids_existentes:
        id_unico = min(ids_existentes)
        if len(ids_existentes) > 1:
            # Fusión: varios grupos anteriores ahora son uno
            fusiones.append({
                'Fecha': datetime.now().strftime("%Y-%m-%d"),
                'ID_Ganador': id_unico,
                'IDs_Fusionados': str(sorted(ids_existentes - {id_unico})),
                'Clientes_Involucrados': str(sorted(comp))
            })
    else:
        id_unico = next_id
        next_id += 1
        grupos_nuevos += 1

    for cliente in comp:
        map_cliente[cliente] = id_unico

df_clean['ID_Cliente_Unico'] = df_clean['Cliente'].map(map_cliente)
df_clean['Es_Duplicado'] = df_clean.groupby('ID_Cliente_Unico')['Cliente'].transform('count') > 1
clientes_unicos_reales = len(componentes)

print(f"\n✓ IDs asignados")
print(f"  Clientes únicos reales:  {clientes_unicos_reales:,}")
print(f"  Grupos nuevos asignados: {grupos_nuevos:,}")
print(f"  Fusiones detectadas:     {len(fusiones):,}")
print(f"  Duplicados encontrados:  {df_clean['Es_Duplicado'].sum():,}")
print(f"  Tasa de deduplicación:   {(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%")

✓ Mapeo existente cargado: 419,974 clientes, max ID: 500,719



✓ IDs asignados
  Clientes únicos reales:  400,726
  Grupos nuevos asignados: 39
  Fusiones detectadas:     1
  Duplicados encontrados:  37,074
  Tasa de deduplicación:   4.59%


## 8. Análisis de Duplicados

In [8]:
# ANÁLISIS DETALLADO DE DUPLICADOS

total_registros = len(df_clean)
total_duplicados = df_clean['Es_Duplicado'].sum()

print("\n📊 Estadísticas de Duplicados:")
print(f"  Total de registros:       {total_registros:,}")
print(f"  Clientes únicos reales:   {clientes_unicos_reales:,}")
print(f"  Registros duplicados:     {total_duplicados:,} ({total_duplicados / total_registros * 100:.2f}% del total)")

print(f"\n📐 Distribución por tamaño de grupo:")
print(f"  {'Tamaño':>8} {'Grupos':>10} {'Registros':>12} {'% del total':>13}")
print(f"  {'-------':>8} {'-------':>10} {'---------':>12} {'-----------':>13}")

group_sizes = df_clean.groupby('ID_Cliente_Unico').size()
size_dist = group_sizes.value_counts().sort_index()

for size, count in size_dist.items():
    registros = count * size
    pct = registros / total_registros * 100
    print(f"  {size:>8} {count:>10,} {registros:>12,} {pct:>12.2f}%")



📊 Estadísticas de Duplicados:
  Total de registros:       420,017
  Clientes únicos reales:   400,726
  Registros duplicados:     37,074 (8.83% del total)

📐 Distribución por tamaño de grupo:
    Tamaño     Grupos    Registros   % del total
   -------    -------    ---------   -----------
         1    382,943      382,943        91.17%
         2     16,246       32,492         7.74%
         3      1,164        3,492         0.83%
         4        209          836         0.20%
         5         19           95         0.02%
         6          7           42         0.01%
         7          3           21         0.00%
         8          2           16         0.00%
        15          1           15         0.00%
        16          1           16         0.00%
        22          1           22         0.01%
        27          1           27         0.01%


## 9. Reordenar Columnas y Preparar Output

In [9]:
# REORDENAR COLUMNAS Y PREPARAR PARA EXPORTAR

df_output = df_clean[[
    'ID_Cliente_Unico',
    'Cliente',
    'Documento',
    'Nombres',
    'Apellidos',
    'Email',
    'Celular'
]].copy()

df_output = df_output.sort_values(['ID_Cliente_Unico', 'Cliente'])

print("✓ Datos preparados para exportar")
print(f"\n📋 Vista final de datos:")
print(df_output.head(20).to_string())

✓ Datos preparados para exportar

📋 Vista final de datos:
        ID_Cliente_Unico    Cliente Documento           Nombres            Apellidos                           Email     Celular
596               100000  C38554676  38554676          PATRICIA              RAMIREZ  claudiaramirez0703@hotmail.com  3148906720
597               100001    C524002    524002             SCHON           BRIAN ADAM           brian.schon@gmail.com  1111111111
598               100002    C137986    137986          VIRGILIO       OSMA NAVARRETE            negado@provenzal.net  3123049875
599               100003  C45467991  45467991    MARIA CONSUELO        GARCIA SUAREZ            negado@provenzal.net  1111111111
600               100004  C38555887  38555887          ANGELICA   TRUJILLO BOCANEGRA           angietb81@hotmail.com  3156252917
601               100005  C52517810  52517810  CLAUDIA PATRICIA      MENDEZ MARTINEZ   notificaciones@flormorado.com  3214905676
602               100006  C20246172  20

## 10. Exportar Resultados

In [10]:
# EXPORTAR A EXCEL Y SQL CON ESTRUCTURA

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
output_file = "Clientes_Unicos.xlsx"
sql_table_clientes = "MASTER_ID"

try:
    # Hoja de resumen estadistico
    resumen = pd.DataFrame({
        'Métrica': [
            'Fecha de ejecución',
            'Registros originales',
            'Clientes únicos reales',
            'Grupos nuevos asignados',
            'Fusiones detectadas',
            'Duplicados encontrados',
            'Tasa de deduplicación',
            'Emails válidos',
            'Celulares válidos',
            f'Celulares excluidos por alta frecuencia (>{MAX_CEL_FREQ} registros)',
            'Uniones por celular (email sentinel o único global)',
            'Grupos celular auditados por esa regla',
            'Grupos con email sentinel en regla celular'
        ],
        'Valor': [
            timestamp,
            len(df_clean),
            clientes_unicos_reales,
            grupos_nuevos,
            len(fusiones),
            df_clean['Es_Duplicado'].sum(),
            f"{(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%",
            (df_clean['Email'] != SENTINEL_EMAIL).sum(),
            (~df_clean['Celular'].isin(CELS_EXCLUIDOS)).sum(),
            len(cels_alta_frecuencia),
            cel_uniones_email_unico,
            cel_grupos_email_unico,
            cel_grupos_con_sentinel
        ]
    })

    # 1) Exportar a Excel (siempre refleja el estado completo actual)
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_output.to_excel(writer, sheet_name='Clientes_Deduplicados', index=False)
        resumen.to_excel(writer, sheet_name='Resumen', index=False)

    # 2) Upsert incremental en SQL: solo inserta clientes nuevos o actualiza fusionados
    clientes_existentes = set(map_existente.keys()) if map_existente else set()

    df_nuevos = df_output[~df_output['Cliente'].isin(clientes_existentes)].copy()

    if map_existente:
        df_fusionados = df_output[
            df_output['Cliente'].isin(clientes_existentes) &
            (df_output['Cliente'].map(map_existente) != df_output['ID_Cliente_Unico'])
        ].copy()
    else:
        df_fusionados = pd.DataFrame()

    if not clientes_existentes:
        df_output.to_sql(sql_table_clientes, con=engine, if_exists='replace', index=False)
        print(f"✓ Primera ejecución: tabla '{sql_table_clientes}' creada con {len(df_output):,} registros")
    else:
        if len(df_nuevos) > 0:
            df_nuevos.to_sql(sql_table_clientes, con=engine, if_exists='append', index=False)
            print(f"✓ {len(df_nuevos):,} clientes nuevos insertados en '{sql_table_clientes}'")
        else:
            print(f"✓ Sin clientes nuevos para insertar")

        if len(df_fusionados) > 0:
            with engine.begin() as conn:
                for _, row in df_fusionados.iterrows():
                    conn.execute(
                        text(f"UPDATE {sql_table_clientes} SET ID_Cliente_Unico = :id WHERE Cliente = :cliente"),
                        {"id": int(row['ID_Cliente_Unico']), "cliente": row['Cliente']}
                    )
            print(f"✓ {len(df_fusionados):,} clientes actualizados por fusión")
        else:
            print(f"✓ Sin actualizaciones por fusión")

    if fusiones:
        print(f"⚠️  {len(fusiones)} fusión(es) detectadas — revisar variable 'fusiones' para detalle")

    print(f"✓ Archivo exportado: {output_file}")

except Exception as e:
    print(f"✗ Error al exportar: {str(e)}")

✓ 42 clientes nuevos insertados en 'MASTER_ID'


✓ 1 clientes actualizados por fusión
⚠️  1 fusión(es) detectadas — revisar variable 'fusiones' para detalle
✓ Archivo exportado: Clientes_Unicos.xlsx


In [11]:
fusiones_df = pd.DataFrame(fusiones)
fusiones_df

,Fecha,ID_Ganador,IDs_Fusionados,Clientes_Involucrados
0,2026-10-14,478076,[480862],"['C900730332', 'C901025359']"


## 11. Resumen Final

In [12]:
# RESUMEN FINAL

print("\n" + "="*60)
print("RESUMEN DE DEDUPLICACIÓN DE CLIENTES")
print("="*60)
print(f"\n📅 Fecha y hora: {timestamp}")
print(f"\n📊 RESULTADOS:")
print(f"  • Registros procesados:    {len(df_clean):,}")
print(f"  • Clientes únicos reales:  {clientes_unicos_reales:,}")
print(f"  • Grupos nuevos asignados: {grupos_nuevos:,}")
print(f"  • Fusiones detectadas:     {len(fusiones):,}")
print(f"  • Duplicados identificados:{df_clean['Es_Duplicado'].sum():,}")
print(f"  • Tasa de deduplicación:   {(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%")
print(f"\n🔗 CONEXIÓN POR ATRIBUTOS:")
print(f"  • Clientes con email válido:   {(df_clean['Email'] != SENTINEL_EMAIL).sum():,}")
print(f"  • Clientes con celular válido: {(~df_clean['Celular'].isin(CELS_EXCLUIDOS)).sum():,}")
print(f"  • Celulares excluidos por alta frecuencia (>{MAX_CEL_FREQ} registros): {len(cels_alta_frecuencia):,} números únicos")
print(f"  • Uniones por celular (email sentinel o único global): {cel_uniones_email_unico:,}")
print(f"\n💾 EXPORTACIÓN:")
print(f"  • Tabla SQL: {sql_table_clientes}")
print(f"  • Archivo Excel: {output_file}")
print(f"  • Registros exportados: {len(df_output):,}")
print("\n" + "="*60)


RESUMEN DE DEDUPLICACIÓN DE CLIENTES

📅 Fecha y hora: 2026-10-14 11:18:09

📊 RESULTADOS:
  • Registros procesados:    420,017
  • Clientes únicos reales:  400,726
  • Grupos nuevos asignados: 39
  • Fusiones detectadas:     1
  • Duplicados identificados:37,074
  • Tasa de deduplicación:   4.59%

🔗 CONEXIÓN POR ATRIBUTOS:
  • Clientes con email válido:   274,045
  • Clientes con celular válido: 313,509
  • Celulares excluidos por alta frecuencia (>4 registros): 201 números únicos
  • Uniones por celular (email sentinel o único global): 9,762

💾 EXPORTACIÓN:
  • Tabla SQL: MASTER_ID
  • Archivo Excel: Clientes_Unicos.xlsx
  • Registros exportados: 420,017

